# ML Model Benchmarks — GPU run (Google Colab)

This notebook runs the benchmarking suite on a **CUDA GPU** so the GPU-specific
metrics (CPU-vs-GPU inference speedup, kernel timing, memory bandwidth) get real
numbers instead of being skipped.

## Before you run
1. **Runtime → Change runtime type → Hardware accelerator → GPU (T4 is fine), then Save.**
2. Run the cells top to bottom (`Runtime → Run all`).

The last cell prints the **GPU vs CPU inference speedup at batch 128**, ready to
paste into the project form. Inference benchmarks build fresh models, so this
does **not** require training and finishes in well under a minute on a T4.

In [ ]:
# 1. Confirm a GPU is actually attached
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'No GPU detected. Runtime > Change runtime type > GPU, then Run all again.'
)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
# 2. Get the code
import os
if not os.path.isdir('ML-Model-Benchmarks'):
    !git clone https://github.com/Maxster360/ML-Model-Benchmarks.git
%cd ML-Model-Benchmarks

In [ ]:
# 3. Install ONLY the deps Colab is missing.
#    Do NOT `pip install -r requirements.txt` here: Colab already ships
#    CUDA-enabled torch/tensorflow, and reinstalling from requirements would
#    replace them with CPU builds and disable the GPU path.
!pip install -q GPUtil tabulate

In [ ]:
# 4a. Fast path — inference benchmarks + GPU profiling only (no training).
#     This is all that's needed for the CPU-vs-GPU speedup number.
!python -m src.benchmark --warmup 10 --repeats 100
!python -m src.gpu_profiler

In [ ]:
# 4b. (Optional) Full pipeline on GPU — trains all 4 models on the complete
#     CIFAR-10 set and regenerates every results file and chart. A few minutes
#     on a T4. Uncomment to run.
# !python main.py

In [ ]:
# 5. Compute the GPU vs CPU inference speedup at batch 128
import json

benchmarks = json.load(open('results/inference_benchmarks.json'))['benchmarks']

def latency(framework, model_type, device, batch_size=128):
    for b in benchmarks:
        if (b['framework'] == framework and b['model_type'] == model_type
                and b['device'] == device and b['batch_size'] == batch_size):
            return b['mean_latency_ms']
    return None

print('Inference speedup (CPU latency / GPU latency) at batch 128:\n')
for fw, cpu_dev, gpu_dev in [('pytorch', 'cpu', 'cuda'), ('tensorflow', 'CPU', 'GPU')]:
    for model_type in ['cnn', 'mlp']:
        cpu_ms = latency(fw, model_type, cpu_dev)
        gpu_ms = latency(fw, model_type, gpu_dev)
        if cpu_ms and gpu_ms:
            print(f'  {fw:<11} {model_type.upper():<3}: '
                  f'CPU {cpu_ms:7.2f} ms | GPU {gpu_ms:6.2f} ms | '
                  f'{cpu_ms / gpu_ms:5.1f}x')
        else:
            print(f'  {fw:<11} {model_type.upper():<3}: missing data '
                  f'(CPU={cpu_ms}, GPU={gpu_ms})')

cnn_cpu = latency('pytorch', 'cnn', 'cpu')
cnn_gpu = latency('pytorch', 'cnn', 'cuda')
if cnn_cpu and cnn_gpu:
    print(f'\nForm value -> GPU vs CPU inference speedup: '
          f'{cnn_cpu / cnn_gpu:.1f}x at batch 128 (PyTorch CNN)')

In [ ]:
# 6. (Optional) Download the regenerated results to your machine
import shutil
from google.colab import files
shutil.make_archive('results_gpu', 'zip', 'results')
files.download('results_gpu.zip')

## After running
- Copy the `Form value -> ...` line from cell 5 into the form.
- If you ran the full pipeline (cell 4b) and want the GPU numbers in the repo,
  download `results_gpu.zip` (cell 6), unzip it over your local `results/`, and
  commit — or send it back and it can be committed for you.